In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import tensorflow as tf
from tensorflow import keras
from keras.models import Model, Sequential
from keras.layers import Input, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Concatenate, LSTM, BatchNormalization, SpatialDropout1D, GlobalAveragePooling1D
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.losses import Huber
import os
import random
from ProfitStrategy import riskless_profit, profits_summary
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt 
import xgboost as xgb
#import tensorflow_addons as tfa

In [33]:
folder = "exported_csvs"

data = {
    os.path.splitext(f)[0]: pd.read_csv(os.path.join(folder, f))
    for f in os.listdir(folder) if f.endswith(".csv")
}

train_idx, val_idx, test_idx = np.array(data['train_indices']).ravel(), np.array(data['val_indices']).ravel() ,np.array(data['test_indices']).ravel()

In [34]:
profit_features = [
    'back_VWAP',
    'lay_VWAP',
    'back_PVT',
    'lay_PVT',
    'WOM', 
    'back_VWAP_log_return', 
    'lay_VWAP_log_return'
]

back_lay_target_feature = 'back_lay_profit_sign'
lay_back_target_feature = 'lay_back_profit_sign'

In [46]:
def evaluate(y_test, y_pred):
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    
    metrics = {
        'f1_score': f1,
        'precision_score' : precision,
        'recall_score': recall,
        'accuracy_score': accuracy
    }
    return metrics

def best_threshold(y_val_true, y_val_probs, r = 0.01):
    thresholds = np.arange(0, 1.01, 0.01)
    scores = []

    for t in thresholds:
        y_val_pred = (y_val_probs >= t).astype(int)
        recall = recall_score(y_val_true, y_val_pred)
        precision = precision_score(y_val_true, y_val_pred) if (y_val_pred.sum() > 0) else 0

        if recall >= r:
            score = precision
        else:
            score = -np.inf

        scores.append(score)

    best_t = thresholds[np.argmax(scores)]
    
    return best_t

def Prepare_Data_XGBoost(selected_features, target_feature, dict_data = data, 
                 train_idx = train_idx, val_idx = val_idx, test_idx = test_idx, window_size=5):
    
    X = np.stack([dict_data[feat].values for feat in selected_features], axis=-1)
    y = np.array(dict_data[target_feature].values)

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    def make_sequences(X, y, w):
        X_seq, y_seq = [], []
        for i in range(X.shape[0]):  
            for t in range(X.shape[1] - w):
                window = X[i, t:t+w, :] #0:5,1:6,...,4:9, so X[i,9,:] is not in X_seq 
                
                min_v = np.min(window, axis=0)
                max_v = np.max(window, axis=0)
                mean_v = np.mean(window, axis=0)
                std_v = np.std(window, axis=0, ddof=1)
                last_v = X[i, t+w-1, :]

                stats_vector = np.concatenate([min_v, max_v, mean_v, std_v, last_v])
                
                X_seq.append(stats_vector)
                y_seq.append(y[i, t+w-1])
                
        return np.array(X_seq), np.array(y_seq)

    X_train, y_train = make_sequences(X_train, y_train, window_size)
    X_val, y_val = make_sequences(X_val, y_val, window_size)
    X_test, y_test = make_sequences(X_test, y_test, window_size)

    return X_train, X_val, X_test, y_train.ravel(), y_val.ravel(), y_test.ravel()

def XGBoost_Results(selected_features, target_feature, dict_data = data, 
              train_idx = train_idx, test_idx = test_idx, 
                    max_depth = 6, verbose = 0):

    X_train, X_val, X_test, y_train, y_val, y_test = Prepare_Data_XGBoost(selected_features,target_feature)

    model = xgb.XGBClassifier(
        max_depth=max_depth,
        learning_rate=0.05,
        n_estimators=1000,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',  
        early_stopping_rounds=20
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=verbose
    )

    y_val_probs = model.predict_proba(X_val)[:, 1]
    y_test_probs = model.predict_proba(X_test)[:, 1]
    
    t = best_threshold(y_val, y_val_probs)
    
    y_pred = (y_test_probs >= t).astype(int)
    
    return {'forecast':y_pred, 
            'model':model, 
            'test_true':y_test,
            'threshold':t,
            'metrics': evaluate(y_test, y_pred),
            'val_probs':y_val_probs,
            'test_probs':y_test_probs}

In [49]:
back_lay_XGBoost_results = XGBoost_Results(profit_features, back_lay_target_feature)
lay_back_XGBoost_results = XGBoost_Results(profit_features, lay_back_target_feature)

In [50]:
back_lay_XGBoost_results['metrics']

{'f1_score': 0.05683510943050304,
 'precision_score': 0.3171355498721228,
 'recall_score': 0.031214600377595974,
 'accuracy_score': 0.809401412527498}

In [51]:
lay_back_XGBoost_results['metrics']

{'f1_score': 0.046640755136035536,
 'precision_score': 0.4315068493150685,
 'recall_score': 0.024652709841518294,
 'accuracy_score': 0.7614449461618618}

In [52]:
back_lay_profit = np.array( data['back_lay_profit'] ) [test_idx]
lay_back_profit = np.array( data['lay_back_profit'] ) [test_idx]

back_lay_profit_seq = []
lay_back_profit_seq = []
w=5

for i in range(back_lay_profit.shape[0]):
    for t in range(back_lay_profit.shape[1] - w):
        back_lay_profit_seq.append(back_lay_profit[i,t+w-1])
        lay_back_profit_seq.append(lay_back_profit[i,t+w-1])
        
back_lay_profit_seq = np.array(back_lay_profit_seq)
lay_back_profit_seq = np.array(lay_back_profit_seq)

In [53]:
print('ideal back_lay profit:', np.sum( back_lay_XGBoost_results['test_true']*back_lay_profit_seq ) )
print('model back_lay profit:', np.sum( back_lay_XGBoost_results['forecast']*back_lay_profit_seq ) )

ideal back_lay profit: 62.4014049193395
model back_lay profit: -5.807481536727655


In [54]:
print('ideal lay_back profit:', np.sum( lay_back_XGBoost_results['test_true']*lay_back_profit_seq ) )
print('model lay_back profit:', np.sum( lay_back_XGBoost_results['forecast']*lay_back_profit_seq ) )

ideal lay_back profit: 71.10556468489756
model lay_back profit: -0.9551280818968605
